# 02 — Exact release dates and bulk acquisition plan

F0 only. Protocol v1.5 + implementation lock v1.5h.


In [11]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from ad_f0.io import load_config, ensure_dirs
cfg=load_config(ROOT); ensure_dirs(ROOT,cfg)
print('ROOT =',ROOT)


ROOT = C:\Users\uysal\AD_ROUTER_CP_F0_v1.8


In [12]:
probe_path=ROOT/'results'/'audit'/'f0_schema_probe.csv'
bridge_path=ROOT/'data'/'interim'/'f0_pre9_target_sequence_bridge.parquet'
assert probe_path.exists() and bridge_path.exists(), 'Run 01_F0_schema_probe.ipynb first.'
probe_df=pd.read_csv(probe_path)
assert set(probe_df.release_label)==set(cfg['chembl']['schema_probe_releases'])
assert probe_df.schema_structural_probe_pass.astype(bool).all()
assert probe_df.primary_semantics_probe_pass.astype(bool).all(), 'Primary semantics preflight failed; bulk acquisition remains blocked.'
print('Three-schema preflight lock passed.')


Three-schema preflight lock passed.


In [13]:
from ad_f0.releases import resolve_exact_release_dates, validate_exact_dates
from ad_f0.archive_download import sqlite_archive_urls
manifest=pd.read_csv(ROOT/cfg['paths']['release_manifest'],dtype={'release':str})
overrides=pd.read_csv(ROOT/cfg['paths']['release_date_overrides'],dtype={'release':str})
resolved=resolve_exact_release_dates(manifest,overrides)
resolved.to_csv(ROOT/'results'/'audit'/'f0_release_date_resolution_audit.csv',index=False)
issues=validate_exact_dates(resolved,cfg['chembl']['require_exact_release_dates'])
if issues:
    for x in issues[:30]: print(' -',x)
    raise RuntimeError('Resolve exact ChEMBL CREATION_DATE values before historical layer assignment. See results/audit/f0_release_date_resolution_audit.csv')

max_primary_date=max(pd.Timestamp(spec['P_end']) for spec in cfg['backtests'].values())
resolved['exact_release_date']=pd.to_datetime(resolved['exact_release_date'])
required=resolved[resolved.exact_release_date<=max_primary_date].copy().sort_values('exact_release_date')
required['official_sqlite_url_1']=[sqlite_archive_urls(v)[0] for v in required.release]
required['official_sqlite_url_2']=[sqlite_archive_urls(v)[1] for v in required.release]
resolved.to_csv(ROOT/'results'/'audit'/'chembl_release_manifest_resolved.csv',index=False,date_format='%Y-%m-%d')
required.to_csv(ROOT/'results'/'audit'/'chembl_release_manifest_required.csv',index=False,date_format='%Y-%m-%d')
required[['release','release_label','exact_release_date','official_sqlite_url_1']].to_csv(
    ROOT/'results'/'audit'/'f0_bulk_download_plan.csv',index=False)
print('Primary F0 requires',len(required),'release snapshots through',max_primary_date.date())
print(required[['release_label','exact_release_date','exact_date_source','exact_date_type']].to_string(index=False))
print('No bulk download was performed. Next: 03_F0_extract_archived_chembl.ipynb')


Primary F0 requires 33 release snapshots through 2023-01-01
release_label exact_release_date                    exact_date_source                    exact_date_type
     CHEMBL01         2009-09-03            ChEMBL chembl_release API                      creation_date
     CHEMBL02         2009-11-30            ChEMBL chembl_release API                      creation_date
     CHEMBL03         2010-04-16            ChEMBL chembl_release API                      creation_date
     CHEMBL04         2010-05-18            ChEMBL chembl_release API                      creation_date
     CHEMBL05         2010-06-24            ChEMBL chembl_release API                      creation_date
     CHEMBL06         2010-08-27            ChEMBL chembl_release API                      creation_date
     CHEMBL07         2010-09-29            ChEMBL chembl_release API                      creation_date
     CHEMBL08         2010-10-26            ChEMBL chembl_release API                      creation_